# Reproduce: Texture Mixer side (TF1-on-TF2 path)

Renders Texture Mixer's native positional blend, target-mean coupling, and
soft-OT coupling for the paper's own first five demo pairs (k=0006, 0009,
0017, 0022, 0053 -- same five `colab/reproduce.ipynb` uses, from the same
shared endpoint images: `texton_matching.data.prepare_demo_endpoints`).
Mirrors what `tm_scale1000_headtohead.ipynb`'s TF phase (cells 1-2)
actually did: the same TF2 + `compat.v1` + `tf_slim` shim, the same
`network-final.pkl` pickle-loading path, the same `tm_prep`/encode/generate
helpers (see `experiments/tm_1000.py`, a direct port of that phase).

**The PASS/FAIL check is pixel-level, not feature-based.** Texture
Mixer's render is deterministic up to GPU/TF nondeterminism (a few grey
levels), so the check compares the freshly-rendered native-lerp midpoint
against the archive's own PNG, `evidence/tm_lerp_demo/<k>.png`, pixel by
pixel (PASS when max <= 8, mean <= 1 grey level out of 255). A DINOv2-based
placement check is printed too, but purely as information, with no
assert -- on a first real run, the pixels matched (max 2-3 grey levels)
while placement was off by 0.05-0.065 on 4/5 pairs, and that gap tracked
the DINOv2/timm version installed at scoring time, not the render. See the
last cell for the full explanation.

**Never imports torch.** The archive is explicit about why: e.g.
`tm_headtohead_otsym.ipynb`: *"POST-RESTART: torch/CNT side. Do NOT run
any TF cell after this."* Mixing TF1-compat and torch/CUDA contexts in one
process is unreliable. This notebook has **exactly one scripted restart**,
between the render cell (TF) and the scoring cell (torch) -- see the
markdown cell in between.

**Restart-safe by design**, same as `reproduce.ipynb`: every cell after
the install cell re-adds `/content/package` to `sys.path` itself and uses
absolute paths, so nothing after the restart depends on a variable from an
earlier cell in a since-cleared kernel.

**Tested end to end on Colab** (T4 GPU): renders reproduced the archive's
PNGs to within 2-3 grey levels. The placement check's DINOv2-version
sensitivity (see above) is why it's informational now rather than a second
pass/fail gate.


In [ ]:
#@title Get the package
import glob, os, shutil

# Download this repository as a zip and upload it through Colab's file panel.
# The upload lands at /<name>.zip (or /content/<name>.zip); any zip name works.
if not os.path.isdir("/content/package"):
    found_zip = next(iter(glob.glob("/*.zip") + glob.glob("/content/*.zip")), None)
    if not found_zip:
        raise RuntimeError(
            "No zip found. Download this repository as a zip and upload it through "
            "Colab's file panel, then run this cell again."
        )
    print(f"found uploaded repository at {found_zip}")
    os.system(f"unzip -qo '{found_zip}' -d /content/package_unzipped")
    src_dir = "/content/package_unzipped"
    entries = os.listdir(src_dir)
    if len(entries) == 1 and os.path.isdir(os.path.join(src_dir, entries[0])):
        src_dir = os.path.join(src_dir, entries[0])
    os.makedirs("/content/package", exist_ok=True)
    for name in os.listdir(src_dir):
        shutil.move(os.path.join(src_dir, name), os.path.join("/content/package", name))
else:
    print("/content/package already present (expected after a restart; disk persists).")

%cd /content/package
!ls

In [ ]:
#@title Install Texture Mixer (TF1-on-TF2 shim; NOT the torch/CNT env)
# No tensorflow version pin: the notebooks never pip-install tensorflow at
# all, they just use whatever's preinstalled on Colab. An upper-bounded pin
# (e.g. "tensorflow<2.16") risks exactly the failure install/cnt.sh had to
# fix for CNT's requirements.in -- pip finding no wheel for your Python.
!pip install -q tf_slim gdown
!bash /content/package/install/texture_mixer.sh /content/TextureMixer


In [ ]:
#@title Get the paper's 5 demo pairs and render every TM rule
import sys, os, json
sys.path.insert(0, "/content/package")

from pathlib import Path
from texton_matching.data import download_dtd, prepare_demo_endpoints

# Same 5 pairs colab/reproduce.ipynb uses (data/pairs_120/pairs.csv,
# k=0006/0009/0017/0022/0053), NOT freshly-sampled ones -- the evidence CSVs
# this notebook checks against were computed for exactly these pairs.
#
# prepare_demo_endpoints is the SAME function colab/reproduce.ipynb's CNT
# side calls, writing to the SAME /content/rendered/A|B/<k>.png layout, so
# both notebooks render from byte-identical endpoint images. (An earlier
# version had this notebook build its own copies under
# tm_scale1000/A|B instead -- a cross-check expecting the shared
# /content/rendered/A|B layout found nothing there, even though this side
# had in fact rendered; tm_1000.py now reads the shared layout directly.)
try:
    dtd_images = download_dtd("/content/dtd")
except Exception as e:
    print(f"DTD download failed ({type(e).__name__}: {e}); "
          f"prepare_demo_endpoints will fall back to the shipped evidence/demo_pairs/ PNGs")
    dtd_images = None

demo = prepare_demo_endpoints("/content/package", "/content/rendered", dtd_images)
import pandas as pd
print(pd.DataFrame(demo).to_string(index=False))

manifest = [dict(k=row["k"], cA=row["cA"], fA=row["fA"], cB=row["cB"], fB=row["fB"]) for row in demo]
Path("/content/rendered/manifest.json").write_text(json.dumps(dict(pairs=manifest)))

!python /content/package/experiments/tm_1000.py \
    --tm-root /content/TextureMixer \
    --manifest /content/rendered/manifest.json \
    --dtd-root /content/dtd/images \
    --out /content/rendered \
    --data-root /content/scored \
    --n-pairs 5


## Restart the session now

`Runtime -> Restart session`, then click the next cell and use
`Runtime -> Run cell and below`.

This is the one restart TF1-compat and torch require between them (see the
markdown cell at the top). Everything the cells above produced -- the
package, Texture Mixer, its checkpoint, the DTD download, and the rendered
midpoints under `/content/rendered` -- is on disk and survives the restart;
only Python state is lost. The scoring cell below re-adds
`/content/package` to `sys.path` itself and re-sets the environment
variable that avoids a Hugging Face secret prompt, so it does not depend on
anything from the cells above having been run in this (fresh) kernel.


In [ ]:
#@title Score native-lerp/mean/soft-OT and print the rows
import os, sys
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"  # see colab/reproduce.ipynb's first cell; re-set after the restart
sys.path.insert(0, "/content/package")

# Absolute paths throughout -- don't rely on the notebook's cwd matching
# /content/package at this point.
!python /content/package/rescore.py --images /content/rendered --data-root /content/scored \
    --dtd-root /content/dtd/images --table table1_tm_1000 --device cuda

import pandas as pd, glob
for f in sorted(glob.glob("/content/scored/jackknife/scores_tm1000_*.csv")):
    print(f); print(pd.read_csv(f).to_string(index=False))


In [ ]:
#@title PASS/FAIL: native-lerp pixels against the shipped evidence PNGs
import sys
sys.path.insert(0, "/content/package")

import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

# This tier's real guarantee is PIXEL reproducibility, not feature-based
# scores: Texture Mixer's render is deterministic up to GPU/TF
# nondeterminism (a few grey levels, typically), so comparing the freshly-
# rendered native-lerp midpoint against the archive's own PNG for the same
# pair is a check this pipeline can actually promise to pass. A DINOv2/timm
# feature-based check (placement, KID, ...) additionally depends on
# whichever exact version of the observer produced the reference numbers --
# that's a real, expected source of drift, not a pipeline bug, and it's
# reported below as information only, not as a pass/fail gate.
evidence_dir = Path("/content/package/evidence/tm_lerp_demo")
manifest = json.loads(Path("/content/rendered/manifest.json").read_text())["pairs"]
check_ks = [p["k"] for p in manifest]
lerp_dir = Path("/content/rendered/jackknife/tm1000/lerp")

MAX_TOL, MEAN_TOL = 8, 1.0
rows = []
all_pass = True
for k in check_ks:
    expected = np.asarray(Image.open(evidence_dir / f"{k}.png").convert("RGB"), dtype=np.int16)
    got = np.asarray(Image.open(lerp_dir / f"{k}.png").convert("RGB"), dtype=np.int16)
    diff = np.abs(expected - got)
    max_diff, mean_diff = int(diff.max()), float(diff.mean())
    ok = max_diff <= MAX_TOL and mean_diff <= MEAN_TOL
    all_pass &= ok
    rows.append(dict(k=k, max_diff=max_diff, mean_diff=round(mean_diff, 4), status="PASS" if ok else "FAIL"))

result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False))
print()
print(f"OVERALL: {'PASS' if all_pass else 'FAIL'} (max <= {MAX_TOL}, mean <= {MEAN_TOL} grey levels/255)")
if not all_pass:
    raise AssertionError("native-lerp pixels did not reproduce the archive's PNGs closely enough -- "
                          "see the diffs above. This IS a pipeline-level check (no observer/feature "
                          "version involved), so a failure here means the render itself differs, not "
                          "just the scorer.")

# --- informational only, below: feature-based placement, not gated on ---
print()
print("--- informational: placement (DINOv2-based; NOT a pass/fail gate -- see note below) ---")
import os
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
from rescore import Extractor, balance_place

evidence_place = pd.read_csv("/content/package/evidence/tm_lerp_place_1000.csv")
evidence_place["k"] = evidence_place["k"].apply(lambda x: f"{int(x):04d}")
evidence_place = evidence_place.set_index("k")

render_dir = Path("/content/rendered/tm_scale1000")
ext = Extractor(device="cuda")
mid_dino = ext.dino_feats([Image.open(lerp_dir / f"{k}.png") for k in check_ks])
a_dino = ext.dino_feats([Image.open(render_dir / "recon_A" / f"{k}.png") for k in check_ks])
b_dino = ext.dino_feats([Image.open(render_dir / "recon_B" / f"{k}.png") for k in check_ks])
computed_place = balance_place(mid_dino, a_dino, b_dino)

info_rows = []
for i, k in enumerate(check_ks):
    expected = float(evidence_place.loc[k, "place"]) if k in evidence_place.index else float("nan")
    got = float(computed_place[i])
    info_rows.append(dict(k=k, expected=expected, computed=got, diff=got - expected))
print(pd.DataFrame(info_rows).to_string(index=False))
print()
print("The placement differences above reflect DINOv2/timm version drift between whatever produced")
print("evidence/tm_lerp_place_1000.csv and the version installed in this session, not a difference in")
print("the rendered pixels (which the PASS/FAIL check above already confirmed match closely).")
